# AI Impact on Student Learning - Model Training & Evaluation (Phase 2)

**Objective**: Train, evaluate, and compare 7 classification algorithms to predict student **Burnout Risk Level** (`Low`, `Medium`, `High`). Perform hyperparameter tuning on the best model, analyze feature importance, and export the trained model pipeline.

## Step 1: Imports and Setup

In [ ]:
import sys
import os
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from src.preprocessor import DataPreprocessor
from src.model_trainer import ModelTrainer
from src.hyperparameter_tuner import HyperparameterTuner
from src.feature_importance import FeatureImportanceAnalyzer

%matplotlib inline
sns.set_theme(style="whitegrid")

## Step 2: Load Cleaned Dataset

In [ ]:
cleaned_data_path = '../data/cleaned_student_data.csv'
df = pd.read_csv(cleaned_data_path)
print(f"Loaded dataset with shape: {df.shape}")
df.head()

## Step 3: Data Preprocessing & Train-Test Split

In [ ]:
preprocessor = DataPreprocessor(target_col="Burnout_Risk_Level")
X_proc, y_proc = preprocessor.fit_transform(df)
X_train, X_test, y_train, y_test = preprocessor.split_data(X_proc, y_proc, test_size=0.2, random_state=42)

target_names = preprocessor.label_encoder.classes_.tolist()
feature_names = preprocessor.feature_names
print(f"Target Classes: {target_names}")

## Step 4: Model Training & 5-Fold Cross-Validation (7 Algorithms)

In [ ]:
trainer = ModelTrainer(output_dir='../reports/figures')
results_df, fitted_models, confusion_matrices, classification_reports = trainer.evaluate_models(
    X_train=X_train, y_train=y_train,
    X_test=X_test, y_test=y_test,
    target_names=target_names
)
results_df

## Step 5: Hyperparameter Tuning via GridSearchCV

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_base_model = fitted_models[best_model_name]

tuner = HyperparameterTuner(best_model_name=best_model_name, base_model=best_base_model)
tuned_model, tuned_metrics = tuner.tune(
    X_train=X_train, y_train=y_train,
    X_test=X_test, y_test=y_test,
    cv=3, scoring="f1_weighted"
)

## Step 6: Feature Importance Analysis

In [ ]:
fi_analyzer = FeatureImportanceAnalyzer(output_dir='../reports/figures')
fi_df = fi_analyzer.analyze(
    model=tuned_model,
    feature_names=feature_names,
    X_test=X_test,
    y_test=y_test,
    top_n=15
)

## Step 7: Export Trained Model Package

In [ ]:
model_save_path = '../models/burnout_prediction_model.pkl'
os.makedirs('../models', exist_ok=True)

model_package = {
    "preprocessor": preprocessor.preprocessor,
    "label_encoder": preprocessor.label_encoder,
    "model": tuned_model,
    "best_model_name": best_model_name,
    "feature_names": feature_names,
    "target_names": target_names,
    "tuned_metrics": tuned_metrics
}
joblib.dump(model_package, model_save_path)
print(f"Trained Model Package successfully saved to: {model_save_path}")